<a href="https://colab.research.google.com/github/RUSHIKETHANA28/2303A51919-B28-RL/blob/main/RL_LAB10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Assignment 10: Bellman Optimality and Value Iteration
# Course: Reinforcement Learning
# Environment: Deterministic 2 x 2 Grid World

# --------------------------------------------------
# 1. Define the Grid World and MDP parameters
# --------------------------------------------------

gamma = 0.9
theta = 1e-6

normal_reward = -1
wall_reward = -1
goal_reward = 10

states = ["S0", "S1", "G"]
non_terminal_states = ["S0", "S1"]
actions = ["Up", "Down", "Left", "Right"]

grid = [
    ["S0", "G"],
    ["W",  "S1"]
]

print("=" * 60)
print("DETERMINISTIC GRID WORLD - MDP DETAILS")
print("=" * 60)

print("Grid World:")
for row in grid:
    print(row)

print("\nStates:", states)
print("Non-terminal states:", non_terminal_states)
print("Terminal state: G")
print("Actions:", actions)
print("Discount factor (gamma):", gamma)
print("Normal cell reward:", normal_reward)
print("Wall reward:", wall_reward)
print("Goal reward:", goal_reward)
print("Convergence threshold (theta):", theta)


# --------------------------------------------------
# 2. Define deterministic transitions
# --------------------------------------------------
# The grid coordinates are:
# S0 = (0, 0), G = (0, 1)
# W  = (1, 0), S1 = (1, 1)
#
# If an action hits a wall or boundary, the agent
# stays in the same state and receives -1.
# Reaching G gives +10. G is terminal.

positions = {
    "S0": (0, 0),
    "S1": (1, 1),
    "G":  (0, 1),
    "W":  (1, 0)
}

directions = {
    "Up":    (-1, 0),
    "Down":  (1, 0),
    "Left":  (0, -1),
    "Right": (0, 1)
}

transition = {}

for state in non_terminal_states:
    r, c = positions[state]

    for action in actions:
        dr, dc = directions[action]
        nr, nc = r + dr, c + dc

        # Check grid boundaries
        if not (0 <= nr < 2 and 0 <= nc < 2):
            next_state = state
            reward = normal_reward

        else:
            cell = grid[nr][nc]

            if cell == "W":
                next_state = state
                reward = wall_reward
            elif cell == "G":
                next_state = "G"
                reward = goal_reward
            else:
                next_state = cell
                reward = normal_reward

        transition[(state, action)] = (next_state, reward)


# --------------------------------------------------
# 3. Verify all state-action transitions
# --------------------------------------------------

print("\n" + "=" * 60)
print("TRANSITION VERIFICATION")
print("=" * 60)

for state in non_terminal_states:
    for action in actions:
        next_state, reward = transition[(state, action)]
        print(
            f"State: {state:2s} | Action: {action:5s} | "
            f"Next state: {next_state:2s} | Reward: {reward:+.1f}"
        )

expected_pairs = len(non_terminal_states) * len(actions)
assert len(transition) == expected_pairs

print(f"\nTotal state-action pairs: {len(transition)}")
print("All state-action pairs have defined transitions.")


# --------------------------------------------------
# 4. Initialize state values
# --------------------------------------------------

V = {state: 0.0 for state in states}
V["G"] = 0.0

print("\n" + "=" * 60)
print("INITIAL STATE VALUES")
print("=" * 60)

for state in states:
    print(f"V({state}) = {V[state]:.4f}")


# --------------------------------------------------
# 5. Implement synchronous Value Iteration
# --------------------------------------------------
# All new values are calculated using the OLD values.
# V_new is not used until the next iteration.

print("\n" + "=" * 60)
print("VALUE ITERATION")
print("=" * 60)

iteration = 0
history = []

print(f"{'Iter':>5} {'V(S0)':>12} {'V(S1)':>12} {'V(G)':>10} {'Delta':>12}")

while True:
    iteration += 1
    V_new = V.copy()

    for state in non_terminal_states:
        q_values = []

        for action in actions:
            next_state, reward = transition[(state, action)]

            q = reward + gamma * V[next_state]
            q_values.append(q)

        V_new[state] = max(q_values)

    # Terminal state remains fixed at zero
    V_new["G"] = 0.0

    # Maximum change across all states
    delta = max(abs(V_new[s] - V[s]) for s in states)

    V = V_new.copy()

    history.append({
        "Iteration": iteration,
        "V(S0)": V["S0"],
        "V(S1)": V["S1"],
        "V(G)": V["G"],
        "Delta": delta
    })

    print(
        f"{iteration:5d} "
        f"{V['S0']:12.6f} "
        f"{V['S1']:12.6f} "
        f"{V['G']:10.6f} "
        f"{delta:12.6f}"
    )

    if delta < theta:
        break

print("\nConvergence achieved!")
print("Number of iterations:", iteration)
print("Convergence threshold:", theta)


# --------------------------------------------------
# 6. Display final optimal state values
# --------------------------------------------------

print("\n" + "=" * 60)
print("FINAL OPTIMAL STATE VALUES")
print("=" * 60)

for state in states:
    print(f"V*({state}) = {V[state]:.6f}")


# --------------------------------------------------
# 7. Calculate optimal action values Q*(s, a)
# --------------------------------------------------

print("\n" + "=" * 60)
print("OPTIMAL ACTION VALUES Q*(s, a)")
print("=" * 60)

Q = {}

for state in non_terminal_states:
    print(f"\nState {state}:")

    for action in actions:
        next_state, reward = transition[(state, action)]

        q_value = reward + gamma * V[next_state]
        Q[(state, action)] = q_value

        print(
            f"Q*({state}, {action:5s}) = {q_value:10.6f} "
            f"(next state = {next_state}, reward = {reward:+.1f})"
        )


# --------------------------------------------------
# 8. Extract the optimal policy
# --------------------------------------------------
# Return ALL actions tied for the maximum Q-value.

print("\n" + "=" * 60)
print("OPTIMAL POLICY")
print("=" * 60)

optimal_policy = {}

for state in non_terminal_states:
    max_q = max(Q[(state, action)] for action in actions)

    best_actions = [
        action for action in actions
        if abs(Q[(state, action)] - max_q) < 1e-9
    ]

    optimal_policy[state] = best_actions

    print(
        f"State {state}: Optimal action(s) = {best_actions}, "
        f"Value = {max_q:.6f}"
    )


# --------------------------------------------------
# 9. Verify the Bellman Optimality equation
# --------------------------------------------------

print("\n" + "=" * 60)
print("BELLMAN OPTIMALITY VERIFICATION")
print("=" * 60)

all_verified = True

for state in non_terminal_states:
    max_q = max(Q[(state, action)] for action in actions)
    difference = abs(V[state] - max_q)
    verified = difference < 1e-9

    if not verified:
        all_verified = False

    print(
        f"State: {state} | V*({state}) = {V[state]:.6f} | "
        f"max Q* = {max_q:.6f} | "
        f"Difference = {difference:.10f} | "
        f"Verified = {verified}"
    )

print("\nOverall Bellman verification:", all_verified)


# --------------------------------------------------
# 10. Summary of results
# --------------------------------------------------

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print("Optimal State Values:", V)
print("Optimal Policy:", optimal_policy)
print("Total Iterations:", iteration)
print("Bellman Equation Verified:", all_verified)


DETERMINISTIC GRID WORLD - MDP DETAILS
Grid World:
['S0', 'G']
['W', 'S1']

States: ['S0', 'S1', 'G']
Non-terminal states: ['S0', 'S1']
Terminal state: G
Actions: ['Up', 'Down', 'Left', 'Right']
Discount factor (gamma): 0.9
Normal cell reward: -1
Wall reward: -1
Goal reward: 10
Convergence threshold (theta): 1e-06

TRANSITION VERIFICATION
State: S0 | Action: Up    | Next state: S0 | Reward: -1.0
State: S0 | Action: Down  | Next state: S0 | Reward: -1.0
State: S0 | Action: Left  | Next state: S0 | Reward: -1.0
State: S0 | Action: Right | Next state: G  | Reward: +10.0
State: S1 | Action: Up    | Next state: G  | Reward: +10.0
State: S1 | Action: Down  | Next state: S1 | Reward: -1.0
State: S1 | Action: Left  | Next state: S1 | Reward: -1.0
State: S1 | Action: Right | Next state: S1 | Reward: -1.0

Total state-action pairs: 8
All state-action pairs have defined transitions.

INITIAL STATE VALUES
V(S0) = 0.0000
V(S1) = 0.0000
V(G) = 0.0000

VALUE ITERATION
 Iter        V(S0)        V(S1) 